# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Release version: {getattr(metadata,'version', None)}")
print(f"Published: {getattr(metadata,'datePublished', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we list all record set `@id`s, their field and column `@id`s for reference.

In [ ]:
# List all record sets and their field/column @id
rs_ids = []
print("Available RecordSets:")
for rs in metadata.record_sets:
    print(f"  RecordSet name: {getattr(rs,'name', None)}   @id: {rs.id}")
    rs_ids.append(rs.id)
    print("    Fields and their columns:")
    for field in rs.fields:
        print(f"      Field: {field.name}   @id: {field.id}")
        if hasattr(field,'column') and field.column is not None:
            # A field can relate to multiple columns
            if isinstance(field.column, list):
                for col in field.column:
                    print(f"        Column: {getattr(col,'name', None)}   @id: {col.id}")
            else:
                print(f"        Column: {getattr(field.column,'name',None)}   @id: {field.column.id}")
    print()
if len(rs_ids)==0:
    print("No record sets found in metadata. Attempting fallback via `dataset.record_set_ids()`...")
    try:
        fallback_rs_ids = dataset.record_set_ids()
        print(f"record_set_ids: {fallback_rs_ids}")
        rs_ids = list(fallback_rs_ids)
    except Exception as e:
        print("No record sets can be found.")
del fallback_rs_ids

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

> We'll show a preview by extracting the first available record set.

In [ ]:
# Extract data from each discovered record set
record_sets = rs_ids  # as discovered above
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for record_set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            dataframes[record_set_id] = pd.DataFrame(records)
        else:
            print(f"  No records in {record_set_id}")
    except Exception as e:
        print(f"  Error loading records from {record_set_id}: {e}")
if len(dataframes)==0:
    raise RuntimeError("No tabular dataframes could be loaded from record sets.")
main_rs_id = list(dataframes.keys())[0]
print(f"Columns in DataFrame for record set {main_rs_id}:")
print(dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Try to automatically find a numeric field from the DataFrame
df = dataframes[main_rs_id]
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
if numeric_columns:
    numeric_field = numeric_columns[0]
else:
    # fallback: try converting common protein numeric fields
    possible_cols = ['MW (kDa)','Abundance','Coverage (%)','Peptide Count','pI']
    found = False
    for col in possible_cols:
        if col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                numeric_field = col
                found = True
                break
            except Exception:
                continue
    if not found:
        raise RuntimeError("No numeric fields available for analysis.")
print(f"Numeric field selected for analysis: {numeric_field}")

threshold = df[numeric_field].quantile(0.75)  # Use top 25% as example threshold
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (top quartile):")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by a categorical field present in the DataFrame
categorical_candidates = [col for col in df.columns if df[col].dtype=='object' and col != numeric_field]
group_field = categorical_candidates[0] if categorical_candidates else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame("Mean_").rename(columns={numeric_field:f"Mean_{numeric_field}"})
    print(f"Grouped data by {group_field} (showing group means):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If group_field exists, show boxplot by group (top N categories)
if group_field:
    top_categories = df[group_field].value_counts().head(5).index.tolist()
    plt.figure(figsize=(10,5))
    subset = df[df[group_field].isin(top_categories)]
    sns.boxplot(x=group_field, y=numeric_field, data=subset)
    plt.title(f"{numeric_field} by {group_field} (top 5 categories)")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides high-dimensional protein information from extracellular vesicles, including fields like molecular weight, abundance, and peptide count.
- We showed how to load Croissant datasets programmatically, inspect the schema by `@id`, and conduct basic EDA.
- Next steps could include feature engineering (using selected columns by `@id`), outlier analysis, downstream modeling, or cross-referencing external datasets via accession numbers.